# Calculate L1 / L2 / Sensor Stats — Gold Layer

**Purpose:** Read `silver_layer.perception_results`, compute three aggregated views,
and write them to Gold Delta tables so the backend poller can fetch pre-computed
results instead of scanning Silver at runtime.

**Gold tables produced:**
- `gold_layer.robot_l1_hourly` — Hourly anomaly density per robot (L1 burst detection)
- `gold_layer.robot_l2_slope` — Linear-regression slope per robot (L2 escalation trend)
- `gold_layer.robot_sensor_stats` — Per-sensor p99/mean scores per robot

In [ ]:
# ── Constants ────────────────────────────────────────────────────────────
L1_DENSITY_THRESHOLD  = 25.0   # anomaly events per hour → burst
L2_MIN_SLOPE          = 0.05   # anomaly-rate change per 6h bucket → escalation
BUCKET_HOURS          = 6

# Per-sensor p99 thresholds from anomaly-detection/checkpoints/threshold_config_v9.json
SENSOR_THRESHOLDS = {
    "wire_feed_speed":  0.13,
    "wire_temperature": 12.06,
    "welding_current":  0.56,
}

# ── Fully-qualified table names ──────────────────────────────────────────
SILVER_TABLE      = "dbw_ted_hackathon_dev_001.silver_layer.perception_results"
GOLD_L1_TABLE     = "dbw_ted_hackathon_dev_001.gold_layer.robot_l1_hourly"
GOLD_L2_TABLE     = "dbw_ted_hackathon_dev_001.gold_layer.robot_l2_slope"
GOLD_SENSOR_TABLE = "dbw_ted_hackathon_dev_001.gold_layer.robot_sensor_stats"

print(f"L1 density threshold : {L1_DENSITY_THRESHOLD}")
print(f"L2 min slope         : {L2_MIN_SLOPE}")
print(f"Sensor thresholds    : {SENSOR_THRESHOLDS}")
print(f"Bucket hours         : {BUCKET_HOURS}")

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, LongType,
    BooleanType, TimestampType, MapType,
)

df = spark.read.table(SILVER_TABLE)

# Normalize robot_code: '0' → 'R-00', '1' → 'R-01', etc.
df = df.withColumn(
    "robot_code",
    F.concat(F.lit("R-"), F.lpad(F.col("robot_code").cast("int"), 2, "0")),
)

df.cache()

total_rows = df.count()
print(f"Silver table rows: {total_rows:,}")
print()
print("Schema:")
df.printSchema()
print()
print("Distinct robot_codes (after normalization):")
df.select("robot_code").distinct().orderBy("robot_code").show(truncate=False)

In [ ]:
from pyspark.sql import functions as F

df_l1 = (
    df
    .groupBy(
        "robot_code",
        F.date_trunc("hour", "observed_at").alias("hour_bucket"),
    )
    .agg(
        F.count("*").alias("total_records"),
        F.sum(F.col("is_file_anomaly").cast("int")).alias("anomaly_count"),
    )
    .withColumn(
        "density_per_hour",
        F.col("anomaly_count").cast("double"),
    )
    .withColumn(
        "is_burst",
        F.col("density_per_hour") >= L1_DENSITY_THRESHOLD,
    )
    .select(
        "robot_code",
        "hour_bucket",
        "total_records",
        "anomaly_count",
        "density_per_hour",
        "is_burst",
    )
    .orderBy("robot_code", "hour_bucket")
)

print(f"L1 rows: {df_l1.count():,}")
df_l1.show(20, truncate=False)

In [ ]:
bucket_seconds = BUCKET_HOURS * 3600

df_buckets = (
    df
    .withColumn(
        "bucket_ts",
        F.floor(F.unix_timestamp("observed_at") / bucket_seconds) * bucket_seconds,
    )
    .groupBy("robot_code", "bucket_ts")
    .agg(
        F.count("*").alias("total_records"),
        F.sum(F.col("is_file_anomaly").cast("int")).alias("anomaly_count"),
    )
    .withColumn(
        "anomaly_rate",
        F.col("anomaly_count") / F.col("total_records"),
    )
    .orderBy("robot_code", "bucket_ts")
)

print(f"Bucket rows: {df_buckets.count():,}")
df_buckets.show(30, truncate=False)

In [ ]:
import numpy as np
from collections import defaultdict
from datetime import datetime, timezone
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, IntegerType, TimestampType
)

# Collect per-robot bucket data (ordered ascending by bucket_ts)
robot_buckets = (
    df_buckets
    .select("robot_code", "bucket_ts", "anomaly_rate")
    .orderBy("robot_code", "bucket_ts")
    .collect()
)

# Group by robot: list of (bucket_ts_unix_secs, anomaly_rate) in ascending time order
robot_data = defaultdict(list)
for row in robot_buckets:
    robot_data[row.robot_code].append((float(row.bucket_ts), float(row.anomaly_rate)))

def _linreg(rates):
    n = len(rates)
    indices = list(range(n))
    x_mean = sum(indices) / n
    y_mean = sum(rates) / n
    num = sum((i - x_mean) * (r - y_mean) for i, r in zip(indices, rates))
    den = sum((i - x_mean) ** 2 for i in indices)
    slope = num / den if den != 0 else 0.0
    ss_tot = sum((r - y_mean) ** 2 for r in rates)
    ss_res = sum((r - (y_mean + slope * (i - x_mean))) ** 2 for i, r in zip(indices, rates))
    r2 = 1.0 - (ss_res / ss_tot) if ss_tot > 0 else 0.0
    return slope, r2

# Rolling accumulation: produce one row per robot per bucket boundary.
# computed_at = end of the last bucket in the window (= bucket_ts + BUCKET_HOURS)
# so that backend query "computed_at <= tick_time" returns the correct snapshot.
# Require at least 2 buckets to compute a meaningful slope.
MIN_BUCKETS = 2
bucket_seconds = BUCKET_HOURS * 3600
l2_rows = []

for robot_code, pts in robot_data.items():
    for cutoff in range(MIN_BUCKETS - 1, len(pts)):
        window = pts[: cutoff + 1]
        rates = [p[1] for p in window]
        last_bucket_start = window[-1][0]  # unix seconds

        slope, r2 = _linreg(rates)

        if slope >= L2_MIN_SLOPE:
            trend = "escalating"
        elif slope <= -L2_MIN_SLOPE:
            trend = "improving"
        else:
            trend = "stable"

        # computed_at = start of next bucket = when this bucket's data is fully available
        computed_at = datetime.utcfromtimestamp(last_bucket_start + bucket_seconds)

        l2_rows.append((robot_code, float(slope), float(r2), trend, len(window), computed_at))

l2_schema = StructType([
    StructField("robot_code",   StringType(),    False),
    StructField("slope",        DoubleType(),    False),
    StructField("r2",           DoubleType(),    False),
    StructField("trend",        StringType(),    False),
    StructField("bucket_count", IntegerType(),   False),
    StructField("computed_at",  TimestampType(), False),
])

df_l2_slope = spark.createDataFrame(l2_rows, schema=l2_schema)
print(f"L2 slope rows: {df_l2_slope.count()}")
df_l2_slope.orderBy("robot_code", "computed_at").show(50, truncate=False)

In [ ]:
from pyspark.sql.types import MapType, StringType, DoubleType
from pyspark.sql.functions import create_map, lit

KNOWN_SENSORS = list(SENSOR_THRESHOLDS.keys())  # ['wire_feed_speed', 'wire_temperature', 'welding_current']

sensor_thresh_map_expr = create_map(*[
    val for kv in SENSOR_THRESHOLDS.items()
    for val in (lit(kv[0]), lit(kv[1]))
])

df_sensor_stats = (
    df
    .withColumn(
        "scores_map",
        F.from_json(
            F.col("per_sensor_scores"),
            MapType(StringType(), DoubleType()),
        ),
    )
    .select(
        "robot_code",
        F.explode("scores_map").alias("sensor_name", "score"),
        "is_file_anomaly",
    )
    # Keep only the 3 calibrated sensors
    .filter(F.col("sensor_name").isin(KNOWN_SENSORS))
    .groupBy("robot_code", "sensor_name")
    .agg(
        F.percentile_approx("score", 0.99).alias("p99_score"),
        F.avg("score").alias("mean_score"),
        F.avg(F.col("is_file_anomaly").cast("double")).alias("anomaly_rate"),
        F.count("*").alias("record_count"),
    )
    .withColumn(
        "sensor_threshold",
        sensor_thresh_map_expr[F.col("sensor_name")],
    )
    .withColumn(
        "is_anomalous_sensor",
        F.col("p99_score") >= F.col("sensor_threshold"),
    )
    .drop("sensor_threshold")
    .orderBy("robot_code", F.desc("p99_score"))
)

print(f"Sensor stats rows: {df_sensor_stats.count()}")
df_sensor_stats.show(50, truncate=False)

In [ ]:
df_l1.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(GOLD_L1_TABLE)

l1_written = spark.read.table(GOLD_L1_TABLE).count()
print(f"Written L1 rows to {GOLD_L1_TABLE}: {l1_written:,}")

In [ ]:
df_l2_slope.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(GOLD_L2_TABLE)

l2_written = spark.read.table(GOLD_L2_TABLE).count()
print(f"Written L2 rows to {GOLD_L2_TABLE}: {l2_written:,}")

In [ ]:
df_sensor_stats.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(GOLD_SENSOR_TABLE)

sensor_written = spark.read.table(GOLD_SENSOR_TABLE).count()
print(f"Written sensor stats rows to {GOLD_SENSOR_TABLE}: {sensor_written:,}")

In [ ]:
print("═" * 60)
print("L1 — Hourly Burst Summary")
print("═" * 60)
spark.read.table(GOLD_L1_TABLE).groupBy("robot_code", "is_burst").count().show(truncate=False)

print()
print("═" * 60)
print("L2 — Slope/Trend Summary")
print("═" * 60)
spark.read.table(GOLD_L2_TABLE).show(truncate=False)

print()
print("═" * 60)
print("Sensor Stats (ordered by p99_score DESC)")
print("═" * 60)
spark.read.table(GOLD_SENSOR_TABLE).orderBy("robot_code", F.desc("p99_score")).show(50, truncate=False)

print()
print("All Gold tables populated successfully.")